# End-to-End Piano Roll Generation

Full pipeline: **Flow model** (L0-L3) → **Inverse PCA** → **HMEP** (L4-L5 prediction) → **Decoder** → Piano roll

In [ ]:
#| default_exp generate

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, pickle, glob, math
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from omegaconf import DictConfig

from midi_rae.core import PatchState, HierarchicalPatchState, EncoderOutput
from midi_rae.swin import SwinDecoder, SwinMaskedEmbeddingPredictor
from midi_rae.train_flow import CrossLevelFlowModel, PerLevelFlowModel, sample_source
from midi_rae.data import EmbeddingDataset
from midi_rae.utils import load_checkpoint, binarize, cjprint

In [ ]:
#| export
def make_grid_pos(n_patches, device='cpu'):
    """Construct (N, 2) grid position tensor for a square patch grid."""
    g = int(n_patches ** 0.5)
    rows, cols = torch.meshgrid(torch.arange(g), torch.arange(g), indexing='ij')
    return torch.stack([rows.flatten(), cols.flatten()], dim=1).float().to(device)

def inverse_pca_level(pca, flat_vec, n_patches, device):
    """flat (n_patches*n_components,) → (1, n_patches, orig_dim) tensor"""
    n_comp = pca.n_components_
    pca_codes = flat_vec.reshape(n_patches, n_comp).cpu().numpy()
    emb_np = pca.inverse_transform(pca_codes)
    return torch.tensor(emb_np, dtype=torch.float32, device=device).unsqueeze(0)

def build_patch_states(flow_vec, pca_models, level_dims, device):
    """Convert flat flow output into a list of PatchState, one per level."""
    states, offset = [], 0
    for i, level_dim in enumerate(level_dims):
        n_patches = level_dim // pca_models[i].n_components_
        emb = inverse_pca_level(pca_models[i], flow_vec[offset:offset+level_dim], n_patches, device)
        pos = make_grid_pos(n_patches, device)
        states.append(PatchState(emb=emb, pos=pos,
                                  non_empty=torch.ones(1, n_patches, device=device),
                                  mae_mask=torch.ones(n_patches, device=device)))
        offset += level_dim
    return states

def batch_patch_states(per_sample):
    """Batch a list of per-sample PatchState lists into one batched list."""
    return [
        PatchState(emb=torch.cat([s[li].emb for s in per_sample], dim=0),
                   pos=per_sample[0][li].pos,
                   non_empty=torch.cat([s[li].non_empty for s in per_sample], dim=0),
                   mae_mask=per_sample[0][li].mae_mask)
        for li in range(len(per_sample[0]))
    ]

def build_enc_out(levels):
    hps = HierarchicalPatchState(levels=levels)
    return EncoderOutput(patches=hps, full_pos=levels[-1].pos,
                         full_non_empty=levels[-1].non_empty,
                         mae_mask=levels[-1].mae_mask)

In [ ]:
#| export
def generate(cfg: DictConfig):
    """Full generation pipeline: Flow → inverse PCA → HMEP → Decoder → piano roll images."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    cjprint(f'Generating on {device}', color='cyan')
    gen = cfg.generate
    flow_cfg = cfg.flow

    # --- infer level_dims from embedding dataset (same as train_flow) ---
    source_scales = list(flow_cfg.source_scales)
    raw_df = flow_cfg.get('source_df', None)
    source_df = list(raw_df) if hasattr(raw_df, '__iter__') else raw_df
    n_levels_flow = len(source_scales)
    levels = [f'L{i}' for i in range(n_levels_flow)]
    if source_df: source_df = source_df[:n_levels_flow]
    paths = sorted(glob.glob(os.path.expandvars(os.path.expanduser(flow_cfg.embedding_glob))))
    assert paths, f'No embedding files found: {flow_cfg.embedding_glob}'
    dataset = EmbeddingDataset(paths[:1], levels=levels)   # load just one chunk to get level_dims
    level_dims = dataset.level_dims
    print(f'  level_dims: {level_dims}')

    # --- load PCA models ---
    pca_dir = Path(os.path.expandvars(os.path.expanduser(gen.pca_dir)))
    pca_models = {}
    for i in range(n_levels_flow):
        with open(pca_dir / f'pca_L{i}_n20.pkl', 'rb') as f:
            pca_models[i] = pickle.load(f)
        print(f'  PCA L{i}: {pca_models[i].n_components_} components, orig_dim={pca_models[i].n_features_in_}')

    # --- load flow model ---
    t_dim = flow_cfg.get('t_dim', 64)
    model_type = flow_cfg.get('model_type', 'per_level')
    self_condition = flow_cfg.get('self_condition', False)
    if model_type == 'cross_level':
        flow_model = CrossLevelFlowModel(level_dims=level_dims, h_dim=flow_cfg.h_dim,
                                          n_layers=flow_cfg.n_layers,
                                          n_attn_layers=flow_cfg.get('n_attn_layers', 2),
                                          n_heads=flow_cfg.get('n_heads', 8),
                                          self_condition=self_condition, t_dim=t_dim)
    else:
        flow_model = PerLevelFlowModel(level_dims=level_dims, h_dim=flow_cfg.h_dim,
                                        n_layers=flow_cfg.n_layers,
                                        self_condition=self_condition, t_dim=t_dim)
    flow_ckpt = os.path.expandvars(os.path.expanduser(gen.flow_ckpt))
    flow_model = load_checkpoint(flow_model, flow_ckpt).to(device).eval()

    # --- sample via Euler integration ---
    n_samples = gen.get('n_samples', 4)
    n_steps   = gen.get('n_steps', 100)
    x = sample_source((n_samples, sum(level_dims)), device=device,
                      source_df=source_df, source_scales=source_scales, level_dims=level_dims)
    dt = 1.0 / n_steps
    with torch.no_grad():
        for step in range(n_steps):
            t = torch.full((n_samples,), step * dt, device=device)
            x = x + flow_model(x, t) * dt
    print(f'  Flow samples: {x.shape}, min={x.min():.2f}, max={x.max():.2f}')

    # --- inverse PCA: flow vectors → patch embeddings ---
    sample_patch_states = [build_patch_states(x[b], pca_models, level_dims, device)
                           for b in range(n_samples)]

    # --- HMEP predicts L4/L5 from generated L0-L3 ---
    m = cfg.model
    n_stages = len(list(m.depths))
    enc_dims = [int(m.embed_dim * 2**(n_stages-1-i)) for i in range(n_stages)]
    hmep = SwinMaskedEmbeddingPredictor(dims=enc_dims)
    hmep_ckpt = os.path.expandvars(os.path.expanduser(gen.hmep_ckpt))
    hmep = load_checkpoint(hmep, hmep_ckpt).to(device).eval()

    fine_levels = list(range(n_levels_flow, n_stages))
    fine_dims   = [enc_dims[li] for li in fine_levels]
    fine_n_patches = [4**li for li in fine_levels]   # L4=4^4=256, L5=4^5=1024

    all_levels = batch_patch_states(sample_patch_states)
    for n_p, dim in zip(fine_n_patches, fine_dims):
        pos = make_grid_pos(n_p, device)
        all_levels.append(PatchState(emb=torch.zeros(n_samples, n_p, dim, device=device),
                                      pos=pos,
                                      non_empty=torch.ones(n_samples, n_p, device=device),
                                      mae_mask=torch.ones(n_p, device=device)))
    with torch.no_grad():
        hmep_preds, _ = hmep(build_enc_out(all_levels), mask_ratio=0)
    print(f'  HMEP preds: {[p.shape for p in hmep_preds]}')

    # replace zero L4/L5 with HMEP predictions
    for j, (n_p, dim) in enumerate(zip(fine_n_patches, fine_dims)):
        li = n_levels_flow + j
        pos = make_grid_pos(n_p, device)
        all_levels[li] = PatchState(emb=hmep_preds[li], pos=pos,
                                     non_empty=torch.ones(n_samples, n_p, device=device),
                                     mae_mask=torch.ones(n_p, device=device))
    enc_out_hmep = build_enc_out(all_levels)

    # also build zero-L4/L5 version for ablation
    all_levels_zero = list(batch_patch_states(sample_patch_states))
    for n_p, dim in zip(fine_n_patches, fine_dims):
        pos = make_grid_pos(n_p, device)
        all_levels_zero.append(PatchState(emb=torch.zeros(n_samples, n_p, dim, device=device),
                                           pos=pos,
                                           non_empty=torch.ones(n_samples, n_p, device=device),
                                           mae_mask=torch.ones(n_p, device=device)))
    enc_out_zero = build_enc_out(all_levels_zero)

    # --- decode ---
    decoder_ckpt = os.path.expandvars(os.path.expanduser(gen.decoder_ckpt))
    decoder = SwinDecoder(img_height=cfg.data.image_size, img_width=cfg.data.image_size,
                           patch_h=m.patch_h, patch_w=m.patch_w,
                           out_channels=cfg.data.in_channels,
                           embed_dim=m.embed_dim, depths=list(m.dec_depths),
                           num_heads=list(m.dec_num_heads), window_size=m.window_size,
                           mlp_ratio=m.mlp_ratio, drop_path_rate=0.0)
    decoder = load_checkpoint(decoder, decoder_ckpt).to(device).eval()

    with torch.no_grad():
        recons      = decoder(enc_out_hmep)
        recons_zero = decoder(enc_out_zero)
    print(f'  Recon shape: {recons.shape}')

    # --- save ---
    out_dir = Path(os.path.expandvars(os.path.expanduser(gen.get('output_dir', 'outputs/generate'))))
    out_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(2, n_samples, figsize=(4*n_samples, 8))
    for i in range(n_samples):
        axes[0, i].imshow(binarize(recons[i, 0]).cpu().numpy(), aspect='auto', origin='lower', cmap='gray_r')
        axes[0, i].set_title(f'HMEP — {i+1}'); axes[0, i].axis('off')
        axes[1, i].imshow(binarize(recons_zero[i, 0]).cpu().numpy(), aspect='auto', origin='lower', cmap='gray_r')
        axes[1, i].set_title(f'Zero L4/L5 — {i+1}'); axes[1, i].axis('off')
    plt.suptitle('Generated Piano Rolls: HMEP L4/L5 (top) vs Zero L4/L5 (bottom)')
    plt.tight_layout()
    out_path = out_dir / 'generated_piano_rolls.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {out_path}')

In [ ]:
#| export
#| eval: false
import hydra

@hydra.main(version_base=None, config_path='../configs', config_name='config_swin')
def generate_main(cfg: DictConfig):
    generate(cfg)

if __name__ == '__main__':
    generate_main()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()